# Adaptive RAG 测试 Notebook

本 notebook 用于测试 Adaptive_RAG.py 的自适应策略。

## 核心特性
- **自适应策略选择**：根据问题类型自动选择最优 RAG 策略
  - 事实题 → Baseline RAG（直接检索）
  - 复杂题/推理题 → Advanced RAG（问题分解 + Rerank）
- **统一文档处理**：chunk_size=600，与 baseline 一致
- **完整评测支持**：集成 Ragas 5 项指标

In [11]:
# 导入 Adaptive RAG
from Adaptive_RAG import (
    AdaptiveRAG,
    run_ragas_evaluation,
    sample_stratified_evaluation
)

# 导入评测集
from evaluation_dataset import eval_questions_50, eval_ground_truths_50

print("✅ 导入成功")

✅ 导入成功


## 1. 初始化 Adaptive RAG 系统

In [12]:
# PDF URL
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"

# 创建系统
rag = AdaptiveRAG(
    pdf_url=pdf_url,
    chunk_size=600,    # 与 baseline 一致
    chunk_overlap=60
)

🚀 初始化 Adaptive RAG 系统
正在加载 PDF: https://arxiv.org/pdf/1706.03762.pdf
✅ 加载完成，共 15 页
✅ 清洗完成，处理 15 页
✅ 切分完成，共 78 个片段
✅ 混合检索器创建完成
✅ Adaptive RAG 系统初始化完成



## 2. 测试不同类型的问题

In [ ]:
# 测试问题
test_questions = [
    # 事实题（应使用 Baseline）
    "Transformer 的 base 配置使用了多少层？",
    "d_model 在 base 配置中取值是多少？",
    
    # 复杂题（应使用 Advanced）
    "Transformer 为什么需要多头注意力机制？",
    "残差连接和层归一化的作用是什么？",
    
    # 推理题（应使用 Advanced）
    "如果去掉残差连接，模型训练会遇到什么问题？",
    "为什么自注意力机制相比循环网络能更好地捕捉长距离依赖？"
]

print("="*80)
print("🧪 测试自适应策略选择")
print("="*80 + "\n")

for question in test_questions:
    answer, strategy, contexts = rag.answer(question)
    print(f"\n{'='*80}")
    print(f"问题: {question}")
    print(f"策略: {strategy}")
    print(f"检索上下文数: {len(contexts)}")
    print(f"答案: {answer}")
    print(f"{'='*80}")

🧪 测试自适应策略选择

📝 [Baseline] 事实题 (keyword): Transformer 的 base 配置使用了多少层？...

问题: Transformer 的 base 配置使用了多少层？
策略: baseline
检索上下文数: 3
答案: 资料中未找到相关内容。
📝 [Baseline] 事实题 (keyword): d_model 在 base 配置中取值是多少？...

问题: d_model 在 base 配置中取值是多少？
策略: baseline
检索上下文数: 3
答案: d_model 在 base 配置中的取值是 512。
🔍 [Advanced] complex题 (keyword): Transformer 为什么需要多头注意力机制？...


## 3. 查看策略使用统计

In [ ]:
# 获取统计信息
stats = rag.get_stats()
total = sum(stats.values())

print("="*80)
print("📊 策略使用统计")
print("="*80)
print(f"事实题 (Baseline): {stats['simple']} ({stats['simple']/total*100:.1f}%)")
print(f"复杂题 (Advanced): {stats['complex']} ({stats['complex']/total*100:.1f}%)")
print(f"推理题 (Advanced): {stats['reasoning']} ({stats['reasoning']/total*100:.1f}%)")
print(f"总计: {total} 题")
print("="*80)

📊 策略使用统计
事实题 (Baseline): 3 (50.0%)
复杂题 (Advanced): 2 (33.3%)
推理题 (Advanced): 1 (16.7%)
总计: 6 题


## 4. 小样本评测（分层采样）

In [ ]:
# 分层采样
eval_indices = sample_stratified_evaluation(total_samples=10)

# 运行评测
sample_report = run_ragas_evaluation(
    adaptive_rag=rag,
    eval_questions=eval_questions_50,
    eval_ground_truths=eval_ground_truths_50,
    indices=eval_indices
)

✅ 分层采样完成：共 10 题
   - 基础知识题: 8 题
   - 复杂题: 1 题
   - 推理题: 1 题
   - 采样题号: [4, 6, 13, 26, 27, 30, 33, 36, 43, 49]

[1/10] Transformer 在每个子层外部使用了哪两种稳定训练的结构？...
📝 [Baseline] 事实题 (keyword): Transformer 在每个子层外部使用了哪两种稳定训练的结构？...
[2/10] 为什么注意力分数要除以 sqrt(d_k)？...
🔍 [Advanced] complex题 (keyword): 为什么注意力分数要除以 sqrt(d_k)？...
[3/10] 编码器-解码器注意力中的 Query、Key、Value 分别来自哪里？...
  🔄 关键词分类置信度较低 (0.30)，使用 LLM Router...
📝 [Baseline] 事实题 (llm_router): 编码器-解码器注意力中的 Query、Key、Value 分别来自哪里？...
[4/10] big 模型的训练耗时大约是多少？...
📝 [Baseline] 事实题 (keyword): big 模型的训练耗时大约是多少？...
[5/10] Transformer base 模型大约有多少参数？...
📝 [Baseline] 事实题 (keyword): Transformer base 模型大约有多少参数？...
[6/10] 论文报告的 Transformer big 在英法任务上的 BLEU 分数是多少？...
📝 [Baseline] 事实题 (keyword): 论文报告的 Transformer big 在英法任务上的 BLEU 分数是多少？...
[7/10] 循环层每层的时间复杂度是多少？...
📝 [Baseline] 事实题 (keyword): 循环层每层的时间复杂度是多少？...
[8/10] 《Attention Is All You Need》主要验证的是哪类任务？...
  🔄 关键词分类置信度较低 (0.30)，使用 LLM Router...
📝 [Baseline] 事实题 (llm_router): 《Attention Is All You Need》主要验证的是哪类任务？..

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:   4%|▍         | 2/50 [00:08<03:09,  3.95s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:   6%|▌         | 3/50 [00:17<04:54,  6.28s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:  30%|███       | 15/50 [00:33<00:52,  1.51s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token 


📊 策略使用统计
事实题 (Baseline): 8 (80.0%)
复杂题 (Advanced): 1 (10.0%)
推理题 (Advanced): 1 (10.0%)


## 5. 查看评测结果

In [ ]:
import pandas as pd

# 显示结果
metric_cols = [
    col for col in [
        "faithfulness",
        "answer_relevancy",
        "context_recall",
        "llm_context_precision_with_reference",
        "answer_correctness"
    ]
    if col in sample_report.columns
]

print("="*80)
print("📊 Adaptive RAG 评测结果")
print("="*80)
print(f"\n{'指标':<30} {'平均分':>10}")
print("-"*50)
print(f"{'Faithfulness (忠实性)':<30} {sample_report['faithfulness'].mean():>10.4f}")
print(f"{'Answer Relevancy (答案相关性)':<30} {sample_report['answer_relevancy'].mean():>10.4f}")
print(f"{'Context Recall (上下文召回)':<30} {sample_report['context_recall'].mean():>10.4f}")
print(f"{'Context Precision (上下文精度)':<30} {sample_report['llm_context_precision_with_reference'].mean():>10.4f}")
print(f"{'Answer Correctness (答案正确性)':<30} {sample_report['answer_correctness'].mean():>10.4f}")
print("-"*50)
print(f"{'总体平均分':<30} {sample_report[metric_cols].mean().mean():>10.4f}")
print("="*80)

# 详细结果
display_cols = ["user_input", "response", "faithfulness", "answer_relevancy", "answer_correctness"]
print("\n详细结果：")
display(sample_report[display_cols].head(10))

📊 Adaptive RAG 评测结果

指标                                    平均分
--------------------------------------------------
Faithfulness (忠实性)                 0.9333
Answer Relevancy (答案相关性)           0.7855
Context Recall (上下文召回)             0.9000
Context Precision (上下文精度)          0.6500
Answer Correctness (答案正确性)         0.6488
--------------------------------------------------
总体平均分                              0.7835

详细结果：


,user_input,response,faithfulness,answer_relevancy,answer_correctness
0,Transformer 在每个子层外部使用了哪两种稳定训练的结构？,Transformer 在每个子层外部使用了残差连接和层归一化。,1.000000,0.722974,0.974363
1,为什么注意力分数要除以 sqrt(d_k)？,注意力分数要除以 sqrt(d_k) 是为了防止点积过大，从而避免 softmax 函数进入...,1.000000,0.965207,0.829187
2,编码器-解码器注意力中的 Query、Key、Value 分别来自哪里？,在编码器-解码器注意力中，Query 来自解码器的输出，Memory Keys 和 Valu...,0.666667,0.921300,0.728627
3,big 模型的训练耗时大约是多少？,big 模型的训练耗时大约是 3.5 天。,1.000000,0.833948,0.724552
4,Transformer base 模型大约有多少参数？,Transformer base 模型有 65 × 10^6 参数。,1.000000,0.833471,0.995971
5,论文报告的 Transformer big 在英法任务上的 BLEU 分数是多少？,论文报告的 Transformer big 在英法任务上的 BLEU 分数是 41.0。,1.000000,0.836321,0.232034
6,循环层每层的时间复杂度是多少？,循环层每层的时间复杂度是O(n · d²)。,1.000000,0.861869,0.997302
7,《Attention Is All You Need》主要验证的是哪类任务？,《Attention Is All You Need》主要验证的任务包括阅读理解、抽象摘要、...,1.000000,0.917472,0.167352
8,受限自注意力在每层的时间复杂度是多少，相比一般自注意力和循环层各有什么优劣？,受限自注意力在每层的时间复杂度是O(r · n · d)。相比一般自注意力的O(n² · d...,0.666667,0.962240,0.652758
9,如果在序列很长的推理场景下运行 Transformer，相比循环网络，主要的性能瓶颈可能在哪里？,最终回答：资料中未明确说明Transformer在长序列推理中的主要性能瓶颈，因此我不知道。,1.000000,0.000000,0.185399


## 6. 全量评测：Baseline vs Adaptive RAG

运行完整 50 题评测，并对 Baseline（已知历史结果）与 Adaptive RAG（当前）进行可视化对比。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 全量 50 题评测 ────────────────────────────────────────────
print("🚀 开始全量 50 题评测，预计耗时 20~40 分钟...\n")
full_report = run_ragas_evaluation(
    adaptive_rag=rag,
    eval_questions=eval_questions_50,
    eval_ground_truths=eval_ground_truths_50
)

# ── 指标列名映射 ──────────────────────────────────────────────
METRIC_COLS = {
    "faithfulness":                          "Faithfulness",
    "answer_relevancy":                      "Answer Relevancy",
    "context_recall":                        "Context Recall",
    "llm_context_precision_with_reference":  "Context Precision",
    "answer_correctness":                    "Answer Correctness",
}

# 提取 Adaptive 实测均值
adaptive_scores = {
    display: full_report[col].mean()
    for col, display in METRIC_COLS.items()
    if col in full_report.columns
}

# ── Baseline 历史基准（来自 rag_baseline.ipynb 已发布结果）────
BASELINE_SCORES = {
    "Faithfulness":       1.000,
    "Answer Relevancy":   0.899,
    "Context Recall":     1.000,
    "Context Precision":  0.850,
    "Answer Correctness": None,   # Baseline 未测此项
}

metrics      = list(METRIC_COLS.values())
baseline_vals = [BASELINE_SCORES.get(m) for m in metrics]
adaptive_vals = [adaptive_scores.get(m) for m in metrics]

# ── 数值汇总表 ────────────────────────────────────────────────
summary = pd.DataFrame({
    "指标":        metrics,
    "Baseline":    [f"{v:.4f}" if v is not None else "—" for v in baseline_vals],
    "Adaptive RAG":[f"{v:.4f}" if v is not None else "—" for v in adaptive_vals],
    "变化":        [
        (f"+{adaptive_vals[i]-baseline_vals[i]:+.4f}"
         if (adaptive_vals[i] is not None and baseline_vals[i] is not None) else "—")
        for i in range(len(metrics))
    ],
})
print("\n" + "="*60)
print("📊 Baseline vs Adaptive RAG · 全量评测对比")
print("="*60)
print(summary.to_string(index=False))
print("="*60)



三种方法对比：


,方法,Faithfulness,Answer Relevancy,Context Recall,Context Precision,Answer Correctness
0,Baseline,1.000000,0.89900,1.000,0.850,NaN
1,Advanced,0.207000,0.86500,0.533,0.167,NaN
2,Adaptive,0.933333,0.78548,0.900,0.650,0.648755


In [ ]:
# ── 可视化：4 图布局 ──────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Baseline vs Adaptive RAG · 全量 50 题 Ragas 评测对比",
             fontsize=15, fontweight="bold", y=0.98)

COLORS = {"Baseline": "#4C72B0", "Adaptive RAG": "#DD8452"}
x      = range(len(metrics))
width  = 0.35

# ── 图1：分组柱状图（五项指标全览）────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
bars_b = ax1.bar(
    [xi - width/2 for xi in x],
    [v if v is not None else 0 for v in baseline_vals],
    width, label="Baseline", color=COLORS["Baseline"], alpha=0.85
)
bars_a = ax1.bar(
    [xi + width/2 for xi in x],
    [v if v is not None else 0 for v in adaptive_vals],
    width, label="Adaptive RAG", color=COLORS["Adaptive RAG"], alpha=0.85
)
for bar in list(bars_b) + list(bars_a):
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                 f"{h:.3f}", ha="center", va="bottom", fontsize=7.5)
ax1.set_xticks(list(x))
ax1.set_xticklabels([m.replace(" ", "\n") for m in metrics], fontsize=8)
ax1.set_ylim(0, 1.2)
ax1.set_ylabel("Score")
ax1.set_title("五项指标全览（分组柱状）", fontsize=10, fontweight="bold")
ax1.legend(fontsize=8)
ax1.axhline(1.0, color="gray", linewidth=0.6, linestyle="--", alpha=0.5)

# ── 图2：Context Precision 单项对比（核心指标）──────────────
ax2 = fig.add_subplot(2, 2, 2)
cp_idx  = metrics.index("Context Precision")
cp_vals = [baseline_vals[cp_idx], adaptive_vals[cp_idx]]
cp_bars = ax2.bar(["Baseline", "Adaptive RAG"], cp_vals,
                  color=[COLORS["Baseline"], COLORS["Adaptive RAG"]],
                  width=0.4, edgecolor="white", linewidth=1.5)
for bar, val in zip(cp_bars, cp_vals):
    if val is not None:
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.015,
                 f"{val:.4f}", ha="center", va="bottom",
                 fontsize=12, fontweight="bold")
delta = adaptive_vals[cp_idx] - baseline_vals[cp_idx]
sign  = "+" if delta >= 0 else ""
ax2.set_ylim(0, 1.2)
ax2.set_ylabel("Score")
ax2.set_title(f"Context Precision 核心对比\n变化：{sign}{delta:.4f}",
              fontsize=10, fontweight="bold")
ax2.axhline(baseline_vals[cp_idx], color=COLORS["Baseline"],
            linewidth=1, linestyle="--", alpha=0.6)

# ── 图3：雷达图（剔除 None 项）──────────────────────────────
radar_metrics = [m for i, m in enumerate(metrics)
                 if baseline_vals[i] is not None and adaptive_vals[i] is not None]
radar_base    = [baseline_vals[metrics.index(m)] for m in radar_metrics]
radar_adap    = [adaptive_vals[metrics.index(m)] for m in radar_metrics]
N = len(radar_metrics)
angles = [n / float(N) * 2 * 3.14159 for n in range(N)]
angles += angles[:1]
radar_base += radar_base[:1]
radar_adap += radar_adap[:1]

ax3 = fig.add_subplot(2, 2, 3, polar=True)
ax3.plot(angles, radar_base, "o-", linewidth=1.5,
         color=COLORS["Baseline"], label="Baseline")
ax3.fill(angles, radar_base, alpha=0.15, color=COLORS["Baseline"])
ax3.plot(angles, radar_adap, "o-", linewidth=1.5,
         color=COLORS["Adaptive RAG"], label="Adaptive RAG")
ax3.fill(angles, radar_adap, alpha=0.15, color=COLORS["Adaptive RAG"])
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels([m.replace(" ", "\n") for m in radar_metrics], fontsize=8)
ax3.set_ylim(0, 1)
ax3.set_yticks([0.25, 0.5, 0.75, 1.0])
ax3.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=6)
ax3.set_title("综合能力雷达图", fontsize=10, fontweight="bold", pad=15)
ax3.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)

# ── 图4：逐指标变化量（Delta 条形图）──────────────────────────
ax4 = fig.add_subplot(2, 2, 4)
delta_metrics = [m for i, m in enumerate(metrics)
                 if baseline_vals[i] is not None and adaptive_vals[i] is not None]
deltas = [adaptive_vals[metrics.index(m)] - baseline_vals[metrics.index(m)]
          for m in delta_metrics]
bar_colors = [COLORS["Adaptive RAG"] if d >= 0 else "#C44E52" for d in deltas]
delta_bars = ax4.barh(delta_metrics, deltas, color=bar_colors, alpha=0.85, height=0.45)
for bar, val in zip(delta_bars, deltas):
    xpos = val + 0.005 if val >= 0 else val - 0.005
    ha   = "left" if val >= 0 else "right"
    ax4.text(xpos, bar.get_y() + bar.get_height()/2,
             f"{val:+.4f}", va="center", ha=ha, fontsize=9, fontweight="bold")
ax4.axvline(0, color="black", linewidth=0.8)
ax4.set_xlabel("Δ Score（Adaptive − Baseline）")
ax4.set_title("各指标相对 Baseline 变化量", fontsize=10, fontweight="bold")
pos_patch = mpatches.Patch(color=COLORS["Adaptive RAG"], alpha=0.85, label="提升")
neg_patch = mpatches.Patch(color="#C44E52",              alpha=0.85, label="下降")
ax4.legend(handles=[pos_patch, neg_patch], fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("ragas_baseline_vs_adaptive.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ 对比图已保存为 ragas_baseline_vs_adaptive.png")
